# 06 — Circuito publicado e activation patching semântico

O notebook 05 estabeleceu o contraste comportamental: em `END`, o GPT-2 Small favorece o token IO ao token S. Aqui usamos DLA e activation patching nas posições semânticas de cada exemplo para diagnosticar as sete classes do circuito IOI publicado em Wang et al. (arXiv:2211.00593).

A lista de 26 nós vem de `Easy-Transformer/easy_transformer/ioi_circuit_extraction.py`: portanto, este notebook **não descobre heads**. Ele valida e visualiza a lista publicada no dataset pareado com N=64, mantendo Name Movers primários e Backup Name Movers separados. O resumo por classe prepara o notebook 07, que testa arestas específicas entre classes já definidas.

In [1]:
import json
import os
from pathlib import Path
import subprocess

import plotly.express as px
import torch

from ioi_replication.paper import (
    PUBLISHED_CIRCUIT,
    CircuitClass,
    SemanticPosition,
    activation_patch_matrix,
    build_paired_dataset,
    logit_difference,
    published_nodes,
)
from ioi_replication.runtime import (
    compatibility_mode_config,
    load_gpt2_small,
    select_device,
)

SEED = 106
N_DEV = 64
# Estas cinco posições retomam os papéis IO, S1, S2, S1+1 e END em cada linha do batch.
SEMANTIC_POSITIONS = tuple(SemanticPosition)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULT_PATH = PROJECT_ROOT / "results/paper/06_circuit_discovery.json"

device = select_device()
model = load_gpt2_small(device)
# pIOI preserva a tarefa; pABC troca os nomes e funciona como fonte de ativações de controle.
dataset = build_paired_dataset(model.tokenizer, n=N_DEV, seed=SEED)
# Ambos têm shape [N_DEV, tokens]; `dataset.positions` selecionará uma posição diferente por exemplo.
ioi_tokens = dataset.ioi_tokens.to(device)
abc_tokens = dataset.abc_tokens.to(device)

# A análise seguinte valida os 26 heads agrupados nas sete classes publicadas, não descobre novos nós.
assert len(published_nodes()) == 26
assert len(PUBLISHED_CIRCUIT) == 7
print(f"device={device}; N={N_DEV}; nós={len(published_nodes())}; classes={len(PUBLISHED_CIRCUIT)}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

device=cuda; N=64; nós=26; classes=7


## Projeção no logit com escala do `ln_final`

Centralizamos cada `hook_result` e aplicamos o `ln_final.hook_scale` específico de cada exemplo e posição antes da projeção na direção IO−S. Isso é **atribuição direta apenas em `END`**, onde o vetor pertence ao residual que alimenta imediatamente o unembedding. Em `IO`, `S1`, `S2` e `S1+1`, reportamos somente uma projeção local escalada: atenção posterior ainda pode transportar e transformar informação entre posições.

In [2]:
# A métrica recebe logits [batch, posição, vocabulário] e lê IO−S no END próprio de cada exemplo.
def metric(logits):
    return logit_difference(
        logits,
        dataset.io_token_ids.to(device),
        dataset.s_token_ids.to(device),
        dataset.positions["END"].to(device),
    )

was_use_attn_result = bool(model.cfg.use_attn_result)
model.set_use_attn_result(True)
try:
    # O cache guarda o resultado de cada head no pIOI e a fonte pABC usada no patching da próxima célula.
    cache_filter = lambda name: (
        name.endswith("attn.hook_result")
        or name == "ln_final.hook_scale"
    )
    ioi_logits, ioi_cache = model.run_with_cache(
        ioi_tokens,
        names_filter=cache_filter,
    )
    _, abc_cache = model.run_with_cache(
        abc_tokens,
        names_filter=lambda name: name.endswith("attn.hook_result"),
    )

    baseline = metric(ioi_logits).detach()
    rows = torch.arange(N_DEV, device=device)
    logit_directions = (
        model.W_U[:, dataset.io_token_ids.to(device)].T
        - model.W_U[:, dataset.s_token_ids.to(device)].T
    )
    final_ln_scale = ioi_cache["ln_final.hook_scale"]
    # A saída terá shape [layer, head, posição semântica], pronta para um mapa por posição.
    logit_projection = torch.empty(
        12, 12, len(SEMANTIC_POSITIONS), device=device
    )
    for layer in range(12):
        result = ioi_cache[f"blocks.{layer}.attn.hook_result"]
        for position_index, position in enumerate(SEMANTIC_POSITIONS):
            semantic_index = dataset.positions[position.value].to(device)
            selected = result[rows, semantic_index]
            centered = selected - selected.mean(dim=-1, keepdim=True)
            scale = final_ln_scale[rows, semantic_index][:, None, :]
            scaled = centered / scale
            logit_projection[layer, :, position_index] = (
                scaled * logit_directions[:, None, :]
            ).sum(dim=-1).mean(dim=0)
finally:
    model.set_use_attn_result(was_use_attn_result)

logit_projection = logit_projection.detach()
assert logit_projection.shape == (12, 12, len(SEMANTIC_POSITIONS))
# Só a fatia END é DLA: nas demais posições, o mapa é uma projeção local ainda sujeita a transporte posterior.
end_axis = SEMANTIC_POSITIONS.index(SemanticPosition.END)
dla_end = logit_projection[:, :, end_axis]
for position_index, position in enumerate(SEMANTIC_POSITIONS):
    is_direct = position is SemanticPosition.END
    title = (
        "DLA IO−S em END"
        if is_direct
        else f"Projeção IO−S local com escala ln_final em {position.value}"
    )
    fig = px.imshow(
        logit_projection[:, :, position_index].float().cpu().numpy(),
        labels={"x": "head", "y": "layer", "color": "atribuição/projeção"},
        title=title,
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="equal",
    )
    fig.update_layout(width=650, height=520)
    fig.show()
print(f"baseline IO−S: {baseline.item():.4f}")

baseline IO−S: 3.6447


## Activation patching `pABC → pIOI`

Para cada célula layer×head×posição, substituímos no forward pIOI somente o `hook_result` daquela cabeça e posição pelo valor correspondente de pABC. O efeito é `métrica_patched − métrica_pIOI`, medido como Δ IO−S bruto em N=64; as 720 intervenções cobrem 12 layers × 12 heads × 5 posições.

A convenção de sinal mudou em relação aos notebooks didáticos. Lá, *recovery* positivo media quanto uma ativação limpa restaurava o comportamento de uma execução corrompida. Aqui fazemos o sentido oposto — inserimos pABC em pIOI —, então Δ negativo indica perda de comportamento útil. Δ positivo também pode ser coerente: ao substituir um Negative Name Mover, removemos parte de sua contribuição contrária a IO e a diferença IO−S pode aumentar.

Não usamos uma *normalized metric*: o efeito permanece na mesma unidade da baseline IO−S.

In [3]:
was_use_attn_result = bool(model.cfg.use_attn_result)
model.set_use_attn_result(True)
try:
    patch_matrix = activation_patch_matrix(
        model=model,
        target_tokens=ioi_tokens,
        source_cache=abc_cache,
        metric=metric,
        target_positions=dataset.positions,
        source_positions=dataset.positions,
        semantic_positions=SEMANTIC_POSITIONS,
        activation="result",
    )
finally:
    model.set_use_attn_result(was_use_attn_result)

# A matriz tem shape [12 layers, 12 heads, 5 posições]; subtrair a baseline produz Δ IO−S bruto.
patch_effect = patch_matrix.values - baseline
assert patch_effect.shape == (12, 12, len(SEMANTIC_POSITIONS))
position_axis = {position: i for i, position in enumerate(SEMANTIC_POSITIONS)}

for position_index, position in enumerate(SEMANTIC_POSITIONS):
    fig = px.imshow(
        patch_effect[:, :, position_index].float().cpu(),
        labels={"x": "head", "y": "layer", "color": "Δ IO−S"},
        title=f"Activation patching pABC→pIOI em {position.value}",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="equal",
    )
    fig.update_layout(width=650, height=520)
    fig.show()

# Reunimos somente o efeito na posição atribuída a cada nó publicado e depois agregamos por classe funcional.
class_summary = {}
for circuit_class, nodes in PUBLISHED_CIRCUIT.items():
    effects = [
        patch_effect[node.layer, node.head, position_axis[node.position]].item()
        for node in nodes
    ]
    class_summary[circuit_class.value] = {
        "n_nodes": len(nodes),
        "mean_patch_effect": sum(effects) / len(effects),
        "nodes": [
            {
                "layer": node.layer,
                "head": node.head,
                "position": node.position.value,
                "patch_effect": effect,
            }
            for node, effect in zip(nodes, effects)
        ],
    }

# Este resumo por classe é a ponte entre os mapas individuais e as arestas testadas no notebook 07.
for circuit_class in CircuitClass:
    summary = class_summary[circuit_class.value]
    print(
        f"{circuit_class.value:22s} n={summary['n_nodes']:2d} "
        f"mean Δ={summary['mean_patch_effect']:+.4f}"
    )

name_movers            n= 3 mean Δ=-0.0247
backup_name_movers     n= 8 mean Δ=-0.0687
negative_name_movers   n= 2 mean Δ=+1.0890
s_inhibition           n= 4 mean Δ=-0.8786
induction              n= 4 mean Δ=-0.4966
duplicate_token        n= 3 mean Δ=-0.3229
previous_token         n= 2 mean Δ=-0.2818


In [4]:
# Os blob IDs ligam o resultado às implementações de circuito, patching e runtime usadas aqui.
source_paths = {
    "patching": "src/ioi_replication/paper/patching.py",
    "circuit": "src/ioi_replication/paper/circuit.py",
    "runtime": "src/ioi_replication/runtime.py",
    "notebook": "notebooks/06_circuit_discovery.ipynb",
}
source_commit = os.environ.get("IOI_SOURCE_COMMIT") or subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
source_blobs = {
    name: subprocess.check_output(
        ["git", "rev-parse", f"{source_commit}:{relative_path}"],
        cwd=PROJECT_ROOT,
        text=True,
    ).strip()
    for name, relative_path in source_paths.items()
}

# O artefato preserva separadamente baseline, DLA em END, projeções locais e efeitos causais do patch.
artifact = {
    "paper": "arXiv:2211.00593",
    "classification": "diagnostic-validation",
    "method": "published-circuit validation with semantic result patching",
    "config": {
        "model_name": "gpt2",
        "seed": SEED,
        "n_examples": N_DEV,
        "dataset_sha256": dataset.dataset_sha256,
        "semantic_positions": [position.value for position in SEMANTIC_POSITIONS],
        "activation": "result",
        "transformer_lens_compatibility_mode": compatibility_mode_config(),
    },
    "traceability": {
        "source_commit": source_commit,
        "hash_method": "git rev-parse <source_commit>:<path> (normalized Git blob IDs)",
        "source_blobs": source_blobs,
    },
    "baseline_logit_difference": baseline.item(),
    "direct_logit_attribution_end": dla_end.float().cpu().tolist(),
    "ln_final_scaled_logit_projection": (
        logit_projection.float().cpu().tolist()
    ),
    "projection_note": (
        "Only END is direct attribution; other semantic positions are "
        "local final-LN-scaled projections."
    ),
    "activation_patched_metric": patch_matrix.values.float().cpu().tolist(),
    "activation_patch_effect": patch_effect.float().cpu().tolist(),
    "published_classes": class_summary,
}
# A célula apenas serializa as leituras já calculadas; o notebook 07 usará a topologia publicada para testar caminhos.
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_PATH.write_text(json.dumps(artifact, indent=2), encoding="utf-8")
print(f"Resultado salvo em {RESULT_PATH.relative_to(PROJECT_ROOT)}")
artifact["config"]

Resultado salvo em results\paper\06_circuit_discovery.json


{'model_name': 'gpt2',
 'seed': 106,
 'n_examples': 64,
 'dataset_sha256': '230498a6bfb41c33a2fc89634a20ff28bd207962bbe420e341dec45cc6fd6b43',
 'semantic_positions': ['IO', 'S1', 'S2', 'S1+1', 'END'],
 'activation': 'result',
 'transformer_lens_compatibility_mode': {'disable_warnings': False,
  'no_processing': False,
  'fold_ln': True,
  'center_writing_weights': True,
  'center_unembed': True,
  'fold_value_biases': True,
  'refactor_factored_attn_matrices': False}}

## Leitura responsável

Os mapas são diagnósticos causais em N=64, não descoberta cega nem reexecução dos 100.000 prompts do artigo. A lista de heads foi publicada previamente e aqui é validada no dataset pareado; sinais ou rankings inesperados neste lote são hipóteses para uma amostra maior, não evidência contra o circuito.

O sinal dos resumos acompanha a intervenção pABC→pIOI. S-Inhibition, Induction e Duplicate/Previous Token apresentam Δ médio negativo, consistente com perda de comportamento útil quando suas ativações são trocadas. Negative Name Movers apresentam Δ positivo, como esperado quando remover uma contribuição contrária a IO melhora IO−S. Já o efeito médio pequeno dos Name Movers primários isolados não implica irrelevância: heads redundantes ou compensação por outros Name Movers podem amortecer uma troca individual.

A separação entre os três Name Movers primários e os oito backups é analítica; juntos, eles preservam os 11 heads agrupados como `name mover` no código histórico. O notebook 07 testa caminhos entre as classes publicadas, e o 08 examina diretamente a compensação pelos Backup Name Movers.